# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** 43  
**Kaggle challenge:** `Deep learning` (either `Classic` or `Deep learning`)  
**Kaggle team name (exact):** "Choco Hunters"  

**Author 1 (sciper):** Ewa Miazga (367059)  
**Author 2 (sciper):** Sameh Lahouar (300454)   
**Author 3 (sciper):** Nour Guermazi (314474) 

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

## 00. Imports

In [1]:
from loader import CustomIAPRDataloader, TrainDataset, ImageOnlyDataset
from models.cnn import SimpleCNN
from helper import get_device, compute_mean_std
from trainer import Trainer
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn


device = get_device()
print(f"Using device: {device}")

Using device: mps


## 00.1 Preprocess dataset with coco

In [2]:
import os
import shutil
import json
import cv2
import pandas as pd
from tqdm import tqdm
from torchvision.datasets import CocoDetection


def extract_patch(image_path, bbox, size=1400):
    x, y, w, h = [int(coord) for coord in bbox]
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Image not found: {image_path}")
    patch = img[y:y+h, x:x+w]
    return patch

def patches_from_coco(source, dest):
    annotations_file = os.path.join(source, "_annotations.coco.json")
    images_dir = source
    patches_dir = os.path.join(dest, "patches")

    if os.path.exists(patches_dir):
        shutil.rmtree(patches_dir)
    os.makedirs(patches_dir)

    data = json.load(open(annotations_file, "r"))

    id_to_label = {e["id"]: e["name"] for e in data["categories"]}
    id_to_images = {e["id"]: e["file_name"] for e in data["images"]}
    annotations = data["annotations"]

    df_labels = pd.DataFrame(columns=["name", "label", "image", "bbox"])

    for i, annotation in tqdm(enumerate(annotations), total=len(annotations)):
        image_id = annotation["image_id"]
        label_id = annotation["category_id"]
        bbox = annotation["bbox"]
        label = id_to_label[label_id]
        image_file = id_to_images[image_id]
        image_path = os.path.join(images_dir, image_file)

        try:
            patch = extract_patch(image_path, bbox)
            idx = str(i).zfill(3)
            patch_path = os.path.join(patches_dir, f"{idx}.jpg")
            cv2.imwrite(patch_path, patch)
            df_labels.loc[i] = [idx, label, image_file, bbox]
        except Exception as e:
            print(f"⚠️ Skipped patch {i} from image {image_file}: {e}")

    df_labels.to_csv(os.path.join(patches_dir, "labels.csv"), index=False)
    print(f"✅ Saved {len(df_labels)} patches and labels to {patches_dir}")

# Cell 2: Define paths and load annotation info
train_img_dir = "dataset_project_iapr2025_coco/train_annotated"
ann_path = os.path.join(train_img_dir, "_annotations.coco.json")

# Load COCO annotations and count classes
with open(ann_path, "r") as f:
    coco_json = json.load(f)
categories = coco_json["categories"]
num_classes = len(categories) + 1  # +1 for background
print("Detected classes:", [c["name"] for c in categories])
print("Total (with background):", num_classes)


# Run it on your dataset
source_path = "dataset_project_iapr2025_coco/train_annotated"
destination_path = "dataset_project_iapr2025_coco/train_patches"
patches_from_coco(source_path, destination_path)

Detected classes: ['objects', 'Amandina', 'Arabia', 'Comtesse', 'Creme_brulee', 'Jelly_Black', 'Jelly_Milk', 'Jelly_White', 'Noblesse', 'Noir_authentique', 'Passion_au_lait', 'Stracciatella', 'Tentation_noir', 'Triangolo']
Total (with background): 15


100%|██████████| 584/584 [00:20<00:00, 27.98it/s]

✅ Saved 584 patches and labels to dataset_project_iapr2025_coco/train_patches/patches


In [6]:
def patches_to_ImageFolder(src, dest):
    # Load the CSV with patch labels
    labels_csv = os.path.join(src, "labels.csv")
    df = pd.read_csv(labels_csv)

    # Clear and recreate the destination folder
    if os.path.exists(dest):
        shutil.rmtree(dest)
    os.makedirs(dest, exist_ok=True)

    # Create subfolders and copy images
    for label in tqdm(df["label"].unique(), desc="Creating folders"):
        label_dir = os.path.join(dest, label)
        os.makedirs(label_dir, exist_ok=True)

        for _, row in df[df["label"] == label].iterrows():
            patch_name = f"{str(row['name']).zfill(3)}.jpg"
            src_file = os.path.join(src, patch_name)
            dest_file = os.path.join(label_dir, patch_name)

            if os.path.exists(src_file):
                shutil.copy(src_file, dest_file)
            else:
                print(f"⚠️ Missing file: {src_file}")

# Run it on your dataset
train_src = os.path.join("dataset_project_iapr2025_coco", "train_patches", "patches")
train_dest = os.path.join("dataset_project_iapr2025_coco", "train_patches", "folder_dataset")
patches_to_ImageFolder(train_src, train_dest)

Creating folders: 100%|██████████| 13/13 [00:00<00:00, 158.70it/s]


In [4]:
from torchvision.datasets import CocoDetection

def get_transform():
    return transforms.Compose([
        transforms.Resize((400, 600)),  # Resize to 1400x1400
        transforms.ToTensor(),  # Converts PIL image or ndarray to tensor
    ])

def collate_fn(batch):
    images, targets = zip(*batch)
    converted_targets = []
    for target in targets:
        boxes = torch.as_tensor([obj['bbox'] for obj in target], dtype=torch.float32)
        boxes[:, 2:] += boxes[:, :2]  # Convert [x,y,w,h] to [x1,y1,x2,y2]
        labels = torch.as_tensor([obj['category_id'] for obj in target], dtype=torch.int64)
        converted_targets.append({'boxes': boxes, 'labels': labels})
    return list(images), converted_targets

# Load dataset
full_dataset = CocoDetection(root=train_img_dir, annFile=ann_path, transform=get_transform())

# Optional: Split into train/val
train_len = int(0.9 * len(full_dataset))
val_len = len(full_dataset) - train_len
train_dataset, val_dataset = random_split(full_dataset, [train_len, val_len])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

# just for now 
test_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
print("Train/Val split:", train_len, "/", val_len)
print("display me first label for train dataset")

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Train/Val split: 81 / 9
display me first label for train dataset


# Patches 

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch

# === 1. Load CSV and create label map ===
csv_path = "dataset_project_iapr2025_coco/train_patches/patches/labels.csv"
image_dir = "dataset_project_iapr2025_coco/train_patches/patches"

df = pd.read_csv(csv_path, dtype={'name': str})
class_names = sorted(df['label'].unique())
class_to_idx = {name: i for i, name in enumerate(class_names)}
df['class_idx'] = df['label'].map(class_to_idx)

# === 2. Train/val split ===
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['class_idx'], random_state=42)

# === 3. Define transform ===
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

# === 4. Define dataset ===
class ChocolatePatchDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_dir, row['name'] + '.jpg')
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['class_idx'], dtype=torch.long)
        return image, label

# === 5. Create datasets and loaders ===
train_dataset = ChocolatePatchDataset(train_df, image_dir, transform)
val_dataset = ChocolatePatchDataset(val_df, image_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# === 6. Check dataset size ===
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
# === 7. Check first label ===
print("First label in train dataset:", train_dataset[0][1].item())

Train dataset size: 467
Validation dataset size: 117


UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('int64'), dtype('<U4')) -> None

In [8]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

class_number = len(full_dataset.coco.cats)
model = SimpleCNN(input_shape=3, hidden_units=64, image_height=128, image_width=128, output_shape=class_number)
loss_fn = nn.CrossEntropyLoss()

#optimizer = torch.optim.SGD(params=model.parameters(), lr=0.05)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

trainer = Trainer(model=model,
                      loss_fn=loss_fn,
                      optimizer=optimizer,
                      scheduler=scheduler,
                      num_epochs=20,
                      train_loader=train_loader,
                      val_loader=val_loader,
                      test_loader=test_loader,
                      device=device)

trainer.train()
#predictions = trainer.predict()
#print(predictions)
#trainer.save_model()
avg_loss, accuracy = trainer.evaluate()
#print(f"1 predition: {predictions[0][0]}")
# print(f"Ground truth: {val_ds[0][1]}")
# print(f"Average Loss: {avg_loss}, Accuracy: {accuracy}%")

/Users/ewamiazga/miniconda3/envs/iapr_project/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('int64'), dtype('<U4')) -> None

In [6]:
avg_loss, accuracy = trainer.evaluate()
#print(f"1 predition: {predictions[0][0]}")
# print(f"Ground truth: {val_ds[0][1]}")
print(f"Average Loss: {avg_loss}, Accuracy: {accuracy}%")

Average Loss: 9.583553513563755, Accuracy: 22.22222222222222%


## 01. Load the dataset

In [6]:
import pandas as pd
# --- Transform: resize and convert to tensor only ---
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

# --- Dataset & Loader ---
csv_path = "dataset_project_iapr2025/train.csv"
image_dir = "dataset_project_iapr2025/train"
df = pd.read_csv(csv_path)
dataset = ImageOnlyDataset(image_dir=image_dir, dataframe=df, transform=transform)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)

mean, std = compute_mean_std(loader)
print(f"\nDataset mean: {mean}")
print(f"Dataset std: {std}")

  0%|          | 0/2 [00:06<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
## Default dataset

transform = transforms.Compose([
    transforms.Resize((200, 300)),  # You can choose a suitable resolution
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

DATASET_DIR = "dataset_project_iapr2025"
loader = CustomIAPRDataloader(base_dir=DATASET_DIR, transform=transform)

full_train_ds = loader.train_dataset 
test_ds = loader.test_dataset
ref_ds = loader.reference_dataset
class_number = loader.class_number
class_names = loader.class_names

train_size = int(0.8 * len(full_train_ds))
val_size = len(full_train_ds) - train_size

train_ds, val_ds = random_split(full_train_ds, [train_size, val_size])

# Display information about the datasets
print(f"Train dataset size: {len(train_ds)}")
print(f"Validation dataset size: {len(val_ds)}")
print(f"Test dataset size: {len(test_ds)}")
print(f"Reference dataset size: {len(ref_ds)}")

# Create DataLoader for training and testing datasets
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

# Example of how to use the DataLoader
for images, labels in train_loader:

    print(f"Batch size: {images.size(0)}")
    print(f"Image shape: {images.shape}")
    print(f"Labels shape: {labels.shape}")
    break  # Remove this to iterate through the entire dataset

# display first image from the train dataset
import matplotlib.pyplot as plt
import numpy as np
def imshow(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# imshow(train_ds[0][0])

Train dataset size: 72
Validation dataset size: 18
Test dataset size: 180
Reference dataset size: 13
Batch size: 8
Image shape: torch.Size([8, 3, 200, 300])
Labels shape: torch.Size([8, 13])


In [ ]:
## 02. Data Augmentation

In [ ]:
from helper import unzip_to

unzip_to("../project/data_project.zip",  "../project/data/")
unzip_to("../project/data_projectv2.zip",  "../project/data/")

In [ ]:
## 02. Data Augmentation

In [ ]:
from helper import unzip_to

unzip_to("../project/data_project.zip",  "../project/data/")
unzip_to("../project/data_projectv2.zip",  "../project/data/")

## 02. Create CNN model
Train, Evaluate and Predict

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

model = SimpleCNN(input_shape=3, hidden_units=64, image_height=200, image_width=300, output_shape=class_number)
loss_fn = nn.MSELoss()
#optimizer = torch.optim.SGD(params=model.parameters(), lr=0.05)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

trainer = Trainer(model=model,
                      loss_fn=loss_fn,
                      optimizer=optimizer,
                      scheduler=scheduler,
                      num_epochs=5,
                      train_loader=train_loader,
                      val_loader=val_loader,
                      test_loader=test_loader,
                      device=device)

trainer.train()
#predictions = trainer.predict()
#print(predictions)
#trainer.save_model()
avg_loss, accuracy = trainer.evaluate()
#print(f"1 predition: {predictions[0][0]}")
print(f"Ground truth: {val_ds[0][1]}")
print(f"Average Loss: {avg_loss}, Accuracy: {accuracy}%")

/Users/ewamiazga/miniconda3/envs/iapr_project/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/5], Loss: 8936360.6419
predicted for first sample tensor([0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0.], device='mps:0')
Epoch [2/5], Loss: 2222.2679
predicted for first sample tensor([1., 1., 1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 0.], device='mps:0')
Epoch [3/5], Loss: 0.8856
predicted for first sample tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='mps:0')
Epoch [4/5], Loss: 0.8824
predicted for first sample tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='mps:0')
Epoch [5/5], Loss: 0.8842
predicted for first sample tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='mps:0')
predicted for first sample tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='mps:0')
Ground truth: tensor([0., 0., 0., 0., 0., 3., 3., 0., 0., 0., 2., 2., 2.])
Average Loss: 1.1348543167114258, Accuracy: 0.0%


In [ ]:
predictions = trainer.predict()
print(f"Predictions: {predictions[0]}")

print(f"Model summary: {model.print_model_summary()}")

Predictions: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0